# Generar Dataset Sintético — SmartProc Copilot

Genera datos realistas de una Telco española (estilo Telefónica/Orange).

**Outputs:**
- `data/synthetic/catalog.csv` — 200 artículos (20 duplicados intencionales)
- `data/synthetic/contracts.json` — 5 contratos marco
- `data/synthetic/transactions.csv` — 2000 transacciones históricas
- `data/synthetic/forecast.csv` — forecast mensual por categoría y comprador
- `data/synthetic/contracts/*.txt` — textos de contratos para RAG

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import os
import random

np.random.seed(42)
random.seed(42)

BASE = os.path.join('..', 'data', 'synthetic')
os.makedirs(BASE, exist_ok=True)
os.makedirs(os.path.join(BASE, 'contracts'), exist_ok=True)
print('Directorios listos:', BASE)

Directorios listos: ../data/synthetic


## 1. Catálogo de artículos — `catalog.csv`

200 artículos de compra de una Telco española: fibra, equipos de red, radio, IT, obra civil, etc.  
20 duplicados intencionales (mismo artículo con nombre/descripción ligeramente diferente).

In [2]:
# --- Definición de artículos base por categoría ---

catalog_base = [
    # FIBRA ÓPTICA
    {"sku": "FO-001", "name": "Cable FO SM G.652D 12 hilos exterior", "description": "Cable de fibra óptica monomodo G.652D 12 hilos para tendido exterior en canalización", "category": "fibra_optica", "unit_price_eur": 1.85, "supplier": "Corning", "uom": "metro"},
    {"sku": "FO-002", "name": "Cable FO SM G.652D 24 hilos exterior", "description": "Cable de fibra óptica monomodo G.652D 24 hilos para tendido exterior en canalización", "category": "fibra_optica", "unit_price_eur": 2.90, "supplier": "Corning", "uom": "metro"},
    {"sku": "FO-003", "name": "Cable FO SM G.652D 48 hilos exterior", "description": "Cable de fibra óptica monomodo G.652D 48 hilos para tendido exterior en canalización", "category": "fibra_optica", "unit_price_eur": 4.20, "supplier": "Prysmian", "uom": "metro"},
    {"sku": "FO-004", "name": "Cable FO SM G.652D 96 hilos exterior", "description": "Cable de fibra óptica monomodo G.652D 96 hilos para tendido exterior en canalización", "category": "fibra_optica", "unit_price_eur": 7.10, "supplier": "Prysmian", "uom": "metro"},
    {"sku": "FO-005", "name": "Cable FO ADSS 24 hilos autosoportado", "description": "Cable fibra óptica ADSS 24 hilos autosoportado para tendido aéreo en postes", "category": "fibra_optica", "unit_price_eur": 3.50, "supplier": "Fujikura", "uom": "metro"},
    {"sku": "FO-006", "name": "Cable FO SM G.657A2 4 hilos interior", "description": "Cable fibra óptica monomodo G.657A2 4 hilos flexible para instalación interior en edificios", "category": "fibra_optica", "unit_price_eur": 0.95, "supplier": "Corning", "uom": "metro"},
    {"sku": "FO-007", "name": "Cable FO SM G.657A2 2 hilos drop FTTH", "description": "Cable drop FTTH monomodo G.657A2 2 hilos para acometida de abonado", "category": "fibra_optica", "unit_price_eur": 0.42, "supplier": "Prysmian", "uom": "metro"},
    {"sku": "FO-008", "name": "Bobina FO SM G.652D 12 hilos 2000m", "description": "Bobina 2000 metros cable fibra óptica monomodo G.652D 12 hilos exterior", "category": "fibra_optica", "unit_price_eur": 3550.0, "supplier": "Corning", "uom": "bobina"},
    {"sku": "FO-009", "name": "Splitter óptico PLC 1:8 SC/APC", "description": "Divisor óptico PLC pasivo 1:8 con conectores SC/APC para red GPON", "category": "fibra_optica", "unit_price_eur": 12.50, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "FO-010", "name": "Splitter óptico PLC 1:16 SC/APC", "description": "Divisor óptico PLC pasivo 1:16 con conectores SC/APC para red GPON", "category": "fibra_optica", "unit_price_eur": 18.90, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "FO-011", "name": "Splitter óptico PLC 1:32 SC/APC", "description": "Divisor óptico PLC pasivo 1:32 con conectores SC/APC para red GPON", "category": "fibra_optica", "unit_price_eur": 28.00, "supplier": "Nokia", "uom": "unidad"},
    {"sku": "FO-012", "name": "Conector SC/APC SM prepulido campo", "description": "Conector SC/APC monomodo para montaje en campo con pulido previo", "category": "fibra_optica", "unit_price_eur": 2.10, "supplier": "Corning", "uom": "unidad"},
    {"sku": "FO-013", "name": "Caja empalme fibra óptica exterior 24FO", "description": "Cierre de empalme exterior para 24 fibras con bandejas y protectores de fusión", "category": "fibra_optica", "unit_price_eur": 45.00, "supplier": "Prysmian", "uom": "unidad"},
    {"sku": "FO-014", "name": "Caja terminal óptica interior 8FO", "description": "Roseta óptica interior para 8 fibras con pigtails SC/APC precableados", "category": "fibra_optica", "unit_price_eur": 22.00, "supplier": "Corning", "uom": "unidad"},
    {"sku": "FO-015", "name": "Tubo microducto HDPE 7/5,5mm", "description": "Microducto HDPE 7mm exterior 5,5mm interior para soplado de fibra óptica", "category": "fibra_optica", "unit_price_eur": 0.28, "supplier": "Dura-Line", "uom": "metro"},

    # EQUIPOS RED ACCESO (FTTH/xDSL)
    {"sku": "RA-001", "name": "ONT Huawei EchoLife EG8145V5 GPON", "description": "Terminal de red óptica GPON 4GE+2POTS+WiFi 2.4/5GHz para abonado residencial", "category": "equipos_red_acceso", "unit_price_eur": 42.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RA-002", "name": "ONT Nokia G-010G-P GPON", "description": "Terminal de red óptica GPON 1GE para abonado residencial básico", "category": "equipos_red_acceso", "unit_price_eur": 28.50, "supplier": "Nokia", "uom": "unidad"},
    {"sku": "RA-003", "name": "ONT Huawei EG8247H5 GPON 4GE+2POTS+WiFi", "description": "Terminal óptico GPON 4GE 2POTS WiFi para clientes residenciales y PYMES", "category": "equipos_red_acceso", "unit_price_eur": 55.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RA-004", "name": "OLT Huawei SmartAX MA5800-X2", "description": "Línea de terminación óptica GPON/XGS-PON 2 slots para central de acceso FTTH", "category": "equipos_red_acceso", "unit_price_eur": 4800.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RA-005", "name": "OLT Nokia 7360 ISAM FX-4", "description": "Concentrador de acceso óptico GPON/XGS-PON 4 slots Nokia para central FTTH", "category": "equipos_red_acceso", "unit_price_eur": 5200.00, "supplier": "Nokia", "uom": "unidad"},
    {"sku": "RA-006", "name": "Tarjeta línea GPON 16 puertos Huawei H901GPSFE", "description": "Tarjeta de línea GPON 16 puertos SC/APC para OLT Huawei MA5800", "category": "equipos_red_acceso", "unit_price_eur": 1850.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RA-007", "name": "Switch acceso L2 24GE+4SFP+ Cisco C1000-24T", "description": "Switch capa 2 24 puertos GE + 4 uplinks SFP+ 10G para armario de distribución", "category": "equipos_red_acceso", "unit_price_eur": 890.00, "supplier": "Cisco", "uom": "unidad"},
    {"sku": "RA-008", "name": "Switch acceso L2 48GE+4SFP+ Cisco C1000-48T", "description": "Switch capa 2 48 puertos GE + 4 uplinks SFP+ 10G para armario de distribución", "category": "equipos_red_acceso", "unit_price_eur": 1350.00, "supplier": "Cisco", "uom": "unidad"},

    # EQUIPOS RED CORE Y TRANSPORTE
    {"sku": "RC-001", "name": "Router core Juniper MX204", "description": "Router core universal 400Gbps con 4 puertos QSFP28 100GE para backbone de red", "category": "equipos_red_core", "unit_price_eur": 38500.00, "supplier": "Juniper", "uom": "unidad"},
    {"sku": "RC-002", "name": "Router PE Cisco ASR 9001", "description": "Router de borde de proveedor Cisco ASR9001 para servicios MPLS corporativos", "category": "equipos_red_core", "unit_price_eur": 45000.00, "supplier": "Cisco", "uom": "unidad"},
    {"sku": "RC-003", "name": "Switch core L3 Huawei CloudEngine CE6870", "description": "Switch datacenter L3 48x25GE+6x100GE para core de red y spine de CPD", "category": "equipos_red_core", "unit_price_eur": 22000.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RC-004", "name": "Plataforma DWDM Nokia 1830 PSS-4", "description": "Sistema de transporte óptico DWDM para enlaces de larga distancia y metro", "category": "equipos_red_core", "unit_price_eur": 68000.00, "supplier": "Nokia", "uom": "unidad"},
    {"sku": "RC-005", "name": "Transceiver SFP+ 10GE SR 850nm", "description": "Módulo óptico SFP+ 10GbE shortreach 850nm multimodo para distancias hasta 300m", "category": "equipos_red_core", "unit_price_eur": 48.00, "supplier": "Cisco", "uom": "unidad"},
    {"sku": "RC-006", "name": "Transceiver SFP+ 10GE LR 1310nm", "description": "Módulo óptico SFP+ 10GbE longreach 1310nm monomodo para distancias hasta 10km", "category": "equipos_red_core", "unit_price_eur": 95.00, "supplier": "Cisco", "uom": "unidad"},
    {"sku": "RC-007", "name": "Transceiver QSFP28 100GE LR4", "description": "Módulo óptico QSFP28 100GbE LR4 monomodo 10km para equipos de core", "category": "equipos_red_core", "unit_price_eur": 850.00, "supplier": "Juniper", "uom": "unidad"},

    # RADIO ACCESO (4G/5G)
    {"sku": "RD-001", "name": "RRU Huawei AAU5613 5G NR 64T64R 3.5GHz", "description": "Unidad de radio activa 5G NR 64T64R banda 3.5GHz para despliegue macro 5G", "category": "radio_acceso", "unit_price_eur": 18500.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RD-002", "name": "RRU Ericsson AIR 6449 5G NR 64T64R 3.5GHz", "description": "Unidad de radio activa Ericsson 5G NR 64T64R banda 3.5GHz para macro", "category": "radio_acceso", "unit_price_eur": 19200.00, "supplier": "Ericsson", "uom": "unidad"},
    {"sku": "RD-003", "name": "BBU Huawei BBU5900 5G/4G", "description": "Unidad de banda base Huawei 5G NR y LTE para centralización de radio en CPD", "category": "radio_acceso", "unit_price_eur": 12000.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RD-004", "name": "BBU Ericsson Baseband 6630", "description": "Unidad de banda base Ericsson para 5G NR y LTE con soporte Cloud RAN", "category": "radio_acceso", "unit_price_eur": 13500.00, "supplier": "Ericsson", "uom": "unidad"},
    {"sku": "RD-005", "name": "RRU Huawei RRU3959 4G LTE 1800MHz", "description": "Unidad de radio remota LTE banda 3 1800MHz 2T2R para cobertura macro 4G", "category": "radio_acceso", "unit_price_eur": 4200.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RD-006", "name": "Antena sectorial 4G/5G tri-banda 65° 18dBi", "description": "Antena pasiva sectorial tri-banda 700/1800/2600 MHz apertura 65° ganancia 18dBi", "category": "radio_acceso", "unit_price_eur": 1800.00, "supplier": "Kathrein", "uom": "unidad"},
    {"sku": "RD-007", "name": "Small cell 5G indoor Huawei LampSite", "description": "Celda pequeña 5G NR para cobertura interior en centros comerciales y oficinas", "category": "radio_acceso", "unit_price_eur": 6500.00, "supplier": "Huawei", "uom": "unidad"},
    {"sku": "RD-008", "name": "Small cell 5G outdoor Nokia AAIB", "description": "Celda pequeña 5G NR exterior para densificación urbana en farolas y mobiliario", "category": "radio_acceso", "unit_price_eur": 7200.00, "supplier": "Nokia", "uom": "unidad"},

    # SERVIDORES Y CPD
    {"sku": "SV-001", "name": "Servidor HPE ProLiant DL380 Gen10 2xXeon", "description": "Servidor rack 2U HPE DL380 Gen10 con 2x Intel Xeon Silver 4214 32GB RAM", "category": "servidores_it", "unit_price_eur": 8500.00, "supplier": "HPE", "uom": "unidad"},
    {"sku": "SV-002", "name": "Servidor Dell PowerEdge R740 2xXeon Gold", "description": "Servidor rack 2U Dell R740 con 2x Xeon Gold 6226R 64GB RAM 2x480GB SSD", "category": "servidores_it", "unit_price_eur": 11200.00, "supplier": "Dell", "uom": "unidad"},
    {"sku": "SV-003", "name": "Servidor HPE ProLiant DL360 Gen10 1xXeon", "description": "Servidor rack 1U HPE DL360 Gen10 con 1x Intel Xeon Silver 4210 16GB RAM", "category": "servidores_it", "unit_price_eur": 5800.00, "supplier": "HPE", "uom": "unidad"},
    {"sku": "SV-004", "name": "Storage HPE MSA 2060 SAN 12Gb", "description": "Array de almacenamiento SAN HPE MSA 2060 12Gb SAS con 24 bahías para CPD", "category": "servidores_it", "unit_price_eur": 18000.00, "supplier": "HPE", "uom": "unidad"},
    {"sku": "SV-005", "name": "Disco SSD 2.5 480GB SATA HPE", "description": "Disco SSD 2.5 pulgadas 480GB SATA 6Gb hot-plug para servidores HPE ProLiant", "category": "servidores_it", "unit_price_eur": 185.00, "supplier": "HPE", "uom": "unidad"},
    {"sku": "SV-006", "name": "Memoria RAM 32GB DDR4 2933MHz ECC HPE", "description": "Módulo RAM 32GB DDR4 2933MHz ECC registrada para servidores HPE Gen10", "category": "servidores_it", "unit_price_eur": 320.00, "supplier": "HPE", "uom": "unidad"},

    # CLIMATIZACIÓN Y ENERGÍA CPD
    {"sku": "CL-001", "name": "CRAC Stulz CyberAir 3PRO 30kW", "description": "Unidad de refrigeración de precisión 30kW para sala de equipos y CPD", "category": "climatizacion_cpd", "unit_price_eur": 28000.00, "supplier": "Stulz", "uom": "unidad"},
    {"sku": "CL-002", "name": "SAI APC Galaxy VM 80kVA", "description": "Sistema de alimentación ininterrumpida trifásico 80kVA para CPD crítico", "category": "climatizacion_cpd", "unit_price_eur": 42000.00, "supplier": "APC", "uom": "unidad"},
    {"sku": "CL-003", "name": "SAI Eaton 9PX 11kVA", "description": "SAI monofásico 11kVA con gestión inteligente de baterías para sala técnica", "category": "climatizacion_cpd", "unit_price_eur": 8500.00, "supplier": "Eaton", "uom": "unidad"},
    {"sku": "CL-004", "name": "PDU rack APC AP8853 trifásica 32A", "description": "Unidad de distribución de energía trifásica 32A con medición por toma para rack", "category": "climatizacion_cpd", "unit_price_eur": 1850.00, "supplier": "APC", "uom": "unidad"},
    {"sku": "CL-005", "name": "Armario rack 42U 600x1000 Legrand", "description": "Rack de 19 pulgadas 42 unidades 600x1000mm con gestión de cables para CPD", "category": "climatizacion_cpd", "unit_price_eur": 1200.00, "supplier": "Legrand", "uom": "unidad"},

    # HERRAMIENTAS Y EPI
    {"sku": "HE-001", "name": "Fusionadora fibra óptica Fujikura 62S+", "description": "Máquina de fusión de fibra óptica por arco eléctrico con alineación de núcleo", "category": "herramientas_epi", "unit_price_eur": 3800.00, "supplier": "Fujikura", "uom": "unidad"},
    {"sku": "HE-002", "name": "OTDR Yokogawa AQ7280 SM/MM", "description": "Reflectómetro óptico en el dominio del tiempo para certificación de instalaciones FO", "category": "herramientas_epi", "unit_price_eur": 12500.00, "supplier": "Yokogawa", "uom": "unidad"},
    {"sku": "HE-003", "name": "Casco de seguridad EN397 con pantalla", "description": "Casco de protección EN397 con pantalla facial abatible para trabajos en altura", "category": "herramientas_epi", "unit_price_eur": 45.00, "supplier": "MSA", "uom": "unidad"},
    {"sku": "HE-004", "name": "Arnés anticaída completo EN361", "description": "Arnés de seguridad anticaída completo homologado EN361 para trabajos en altura", "category": "herramientas_epi", "unit_price_eur": 185.00, "supplier": "MSA", "uom": "unidad"},
    {"sku": "HE-005", "name": "Guantes dieléctricos clase 0 EN60903", "description": "Guantes aislantes clase 0 1000V AC para trabajos eléctricos en baja tensión", "category": "herramientas_epi", "unit_price_eur": 38.00, "supplier": "MSA", "uom": "par"},
    {"sku": "HE-006", "name": "Analizador de redes Fluke DSX2-8000", "description": "Certificador de instalaciones LAN Cat6A/Cat8 y fibra óptica hasta 2GHz", "category": "herramientas_epi", "unit_price_eur": 9800.00, "supplier": "Fluke", "uom": "unidad"},

    # OBRA CIVIL Y CANALIZACIÓN
    {"sku": "OC-001", "name": "Tubo PVC corrugado doble pared DN110", "description": "Tubo de canalización PVC corrugado doble pared DN110 para instalación soterrada", "category": "obra_civil", "unit_price_eur": 3.80, "supplier": "Tubos Agua", "uom": "metro"},
    {"sku": "OC-002", "name": "Tubo PEAD doble pared 63mm", "description": "Tubo PEAD corrugado doble pared 63mm para canalización soterrada de telecomunicaciones", "category": "obra_civil", "unit_price_eur": 1.95, "supplier": "Dura-Line", "uom": "metro"},
    {"sku": "OC-003", "name": "Arqueta telecomunicaciones tipo H hormigón", "description": "Arqueta prefabricada de hormigón tipo H para registro de telecomunicaciones en acera", "category": "obra_civil", "unit_price_eur": 280.00, "supplier": "Prehorsa", "uom": "unidad"},
    {"sku": "OC-004", "name": "Poste galvanizado telecomunicaciones 9m", "description": "Poste metálico galvanizado 9 metros para tendido aéreo de cables de telecomunicaciones", "category": "obra_civil", "unit_price_eur": 420.00, "supplier": "Tubos Reunidos", "uom": "unidad"},
    {"sku": "OC-005", "name": "Cinta señalizadora cable enterrado", "description": "Cinta plástica señalizadora amarilla 150mm para señalización de cables enterrados", "category": "obra_civil", "unit_price_eur": 0.15, "supplier": "Tubos Agua", "uom": "metro"},

    # CABLES DE COBRE
    {"sku": "CU-001", "name": "Cable par trenzado Cat6A U/FTP 4 pares", "description": "Cable de red Cat6A U/FTP 4 pares 23AWG para instalaciones LAN de alta velocidad", "category": "cables_cobre", "unit_price_eur": 0.85, "supplier": "Prysmian", "uom": "metro"},
    {"sku": "CU-002", "name": "Cable par trenzado Cat6 U/UTP 4 pares", "description": "Cable de red Cat6 U/UTP 4 pares 23AWG para instalaciones LAN estándar", "category": "cables_cobre", "unit_price_eur": 0.52, "supplier": "Prysmian", "uom": "metro"},
    {"sku": "CU-003", "name": "Cable coaxial RG-6 75 ohm", "description": "Cable coaxial RG-6 75 ohmios para distribución de señal de TV en edificios", "category": "cables_cobre", "unit_price_eur": 0.48, "supplier": "Prysmian", "uom": "metro"},
    {"sku": "CU-004", "name": "Cable tierra 35mm2 verde/amarillo", "description": "Cable unipolar 35mm2 verde amarillo para conexiones de tierra en instalaciones", "category": "cables_cobre", "unit_price_eur": 4.20, "supplier": "Prysmian", "uom": "metro"},
    {"sku": "CU-005", "name": "Cable alimentación 2x2,5mm2 H07V-K", "description": "Cable flexible bipolar 2x2,5mm2 H07V-K para alimentación de equipos en rack", "category": "cables_cobre", "unit_price_eur": 1.10, "supplier": "Prysmian", "uom": "metro"},
]

print(f'Artículos base definidos: {len(catalog_base)}')

Artículos base definidos: 65


In [3]:
# --- Generar 180 artículos únicos + 20 duplicados intencionales ---

# Ampliar catálogo con variantes realistas hasta llegar a 180 únicos
additional_items = [
    # Más fibra
    {"sku": "FO-016", "name": "Pigtail SC/APC SM 2m", "description": "Latigillo de fibra óptica SC/APC monomodo 2 metros para parcheo en rack", "category": "fibra_optica", "unit_price_eur": 3.50, "supplier": "Corning", "uom": "unidad"},
    {"sku": "FO-017", "name": "Patch cord SC/APC-SC/APC SM 3m", "description": "Latiguillo dúplex SC/APC a SC/APC monomodo 3 metros para repartidor óptico", "category": "fibra_optica", "unit_price_eur": 5.20, "supplier": "Corning", "uom": "unidad"},
    {"sku": "FO-018", "name": "Repartidor óptico ODF 24FO rack 1U", "description": "Panel de parcheo óptico 24 SC/APC adaptadores en rack 19 pulgadas 1U", "category": "fibra_optica", "unit_price_eur": 185.00, "supplier": "Corning", "uom": "unidad"},
    {"sku": "FO-019", "name": "Repartidor óptico ODF 48FO rack 2U", "description": "Panel de parcheo óptico 48 SC/APC adaptadores en rack 19 pulgadas 2U", "category": "fibra_optica", "unit_price_eur": 310.00, "supplier": "Prysmian", "uom": "unidad"},
    {"sku": "FO-020", "name": "Cable FO SM G.652D 144 hilos exterior", "description": "Cable fibra óptica monomodo G.652D 144 hilos para troncales de red metropolitana", "category": "fibra_optica", "unit_price_eur": 11.50, "supplier": "Prysmian", "uom": "metro"},
    # Más red acceso
    {"sku": "RA-009", "name": "ONT ZTE F670L GPON 4GE+WiFi", "description": "Terminal óptico GPON ZTE 4GE con WiFi doble banda para abonados residenciales", "category": "equipos_red_acceso", "unit_price_eur": 35.00, "supplier": "ZTE", "uom": "unidad"},
    {"sku": "RA-010", "name": "Switch distribución L3 48GE+4SFP+ HPE 2930F", "description": "Switch de distribución L3 HPE 2930F 48 puertos GE 4 SFP+ para edificios", "category": "equipos_red_acceso", "unit_price_eur": 2800.00, "supplier": "HPE", "uom": "unidad"},
    # Más radio
    {"sku": "RD-009", "name": "RRU Ericsson Radio 4480 4G LTE 2600MHz", "description": "Unidad radio remota Ericsson LTE banda 7 2600MHz 4T4R para macro 4G", "category": "radio_acceso", "unit_price_eur": 5800.00, "supplier": "Ericsson", "uom": "unidad"},
    {"sku": "RD-010", "name": "Cable AISG Ericsson RET 10m", "description": "Cable AISG para control remoto de inclinación eléctrica de antenas Ericsson", "category": "radio_acceso", "unit_price_eur": 95.00, "supplier": "Ericsson", "uom": "unidad"},
    {"sku": "RD-011", "name": "Conector DIN 7/16 macho recto", "description": "Conector RF DIN 7/16 macho recto para feeder coaxial en emplazamientos radio", "category": "radio_acceso", "unit_price_eur": 18.50, "supplier": "Huber+Suhner", "uom": "unidad"},
    {"sku": "RD-012", "name": "Cable feeder coaxial 7/8 50 ohm", "description": "Cable coaxial rígido 7/8 pulgadas 50 ohmios para alimentación de antenas de radio", "category": "radio_acceso", "unit_price_eur": 22.00, "supplier": "Huber+Suhner", "uom": "metro"},
    # Más IT
    {"sku": "SV-007", "name": "Switch ToR Dell EMC N3248TE 48x25GE", "description": "Switch Top-of-Rack 48x25GE+8x100GE para spine-leaf en CPD", "category": "servidores_it", "unit_price_eur": 28000.00, "supplier": "Dell", "uom": "unidad"},
    {"sku": "SV-008", "name": "Disco HDD 3.5 4TB SAS 7.2k HPE", "description": "Disco duro SAS 3.5 pulgadas 4TB 7200rpm hot-plug para NAS y servidores HPE", "category": "servidores_it", "unit_price_eur": 210.00, "supplier": "HPE", "uom": "unidad"},
    {"sku": "SV-009", "name": "Licencia VMware vSphere 8 Standard", "description": "Licencia VMware vSphere 8 Standard por socket con 1 año de soporte básico", "category": "servidores_it", "unit_price_eur": 1850.00, "supplier": "VMware", "uom": "licencia"},
    # Más obra civil
    {"sku": "OC-006", "name": "Grapa acero inox galvanizado para poste", "description": "Grapa de sujeción acero inoxidable galvanizado para fijación de cables en postes", "category": "obra_civil", "unit_price_eur": 2.80, "supplier": "Tubos Reunidos", "uom": "unidad"},
    {"sku": "OC-007", "name": "Tapa registro H hormigón 60x60", "description": "Tapa de registro 60x60cm en hormigón armado con marco metálico para acera", "category": "obra_civil", "unit_price_eur": 95.00, "supplier": "Prehorsa", "uom": "unidad"},
    # Más EPI
    {"sku": "HE-007", "name": "Linterna frontal LED 500lm ATEX", "description": "Linterna frontal LED 500 lúmenes certificación ATEX para zonas con riesgo explosión", "category": "herramientas_epi", "unit_price_eur": 125.00, "supplier": "Peli", "uom": "unidad"},
    {"sku": "HE-008", "name": "Medidor de potencia óptica FO -70 a +6dBm", "description": "Fotómetro para medición de potencia óptica rango -70 a +6dBm para SM y MM", "category": "herramientas_epi", "unit_price_eur": 280.00, "supplier": "Fluke", "uom": "unidad"},
    # Más cables cobre
    {"sku": "CU-006", "name": "Patch cord RJ45 Cat6A 1m azul", "description": "Latiguillo de red RJ45 Cat6A S/FTP 1 metro color azul para patcheo en rack", "category": "cables_cobre", "unit_price_eur": 3.20, "supplier": "Prysmian", "uom": "unidad"},
    {"sku": "CU-007", "name": "Patch cord RJ45 Cat6A 3m rojo", "description": "Latiguillo de red RJ45 Cat6A S/FTP 3 metros color rojo para patcheo en rack", "category": "cables_cobre", "unit_price_eur": 4.50, "supplier": "Prysmian", "uom": "unidad"},
    # Climatización
    {"sku": "CL-006", "name": "CRAC Stulz CyberAir 3PRO 60kW", "description": "Unidad de refrigeración de precisión 60kW para sala principal de CPD", "category": "climatizacion_cpd", "unit_price_eur": 52000.00, "supplier": "Stulz", "uom": "unidad"},
    {"sku": "CL-007", "name": "Batería de repuesto SAI APC SRBT4", "description": "Módulo de baterías de repuesto para SAI APC Galaxy series autonomía extendida", "category": "climatizacion_cpd", "unit_price_eur": 3200.00, "supplier": "APC", "uom": "unidad"},
    # Red acceso
    {"sku": "RA-011", "name": "OLT ZTE C320 GPON/XGS-PON", "description": "Concentrador óptico ZTE C320 GPON y XGS-PON para central de acceso FTTH", "category": "equipos_red_acceso", "unit_price_eur": 4200.00, "supplier": "ZTE", "uom": "unidad"},
    {"sku": "RA-012", "name": "Tarjeta GPON 16p ZTE GTGO", "description": "Tarjeta de línea GPON 16 puertos para OLT ZTE C320", "category": "equipos_red_acceso", "unit_price_eur": 1500.00, "supplier": "ZTE", "uom": "unidad"},
]

all_unique = catalog_base + additional_items

# Rellenar hasta 180 únicos con variantes de precio/proveedor
extra_items = []
for i in range(180 - len(all_unique)):
    base = all_unique[i % len(all_unique)].copy()
    base['sku'] = f"EXT-{i+1:03d}"
    base['unit_price_eur'] = round(base['unit_price_eur'] * np.random.uniform(0.9, 1.1), 2)
    extra_items.append(base)

unique_items = all_unique + extra_items
unique_items = unique_items[:180]
print(f'Artículos únicos: {len(unique_items)}')

Artículos únicos: 180


In [4]:
# --- 20 duplicados intencionales (mismo artículo, nombre/descripción diferente) ---
# Simulan el problema real de maverick purchases: comprar lo mismo con distintos nombres

duplicate_pairs = [
    # (sku_base, nombre_alternativo, descripcion_alternativa, proveedor_alternativo, precio_diferente)
    ("FO-001", "FO-DUP-001", "Fibra óptica SM 12F exterior canalización",
     "Cable fibra monomodo 12 fibras G652D para canalización exterior enterrada", "Corning", 2.05),
    ("FO-002", "FO-DUP-002", "FO monomodo 24 fibras exterior",
     "Fibra óptica monomodo 24H estándar G.652D para uso exterior en canaleta", "Prysmian", 3.10),
    ("FO-007", "FO-DUP-003", "Drop óptico 2FO para abonado FTTH",
     "Cable de acometida drop 2 fibras monomodo para conexión abonado final GPON", "Corning", 0.50),
    ("FO-009", "FO-DUP-004", "Divisor óptico 1x8 SC-APC",
     "Splitter PLC pasivo para red PON 1 entrada 8 salidas conector SC/APC", "Nokia", 14.00),
    ("RA-001", "RA-DUP-001", "Módulo ONT GPON Huawei EG8145",
     "ONT Huawei 4 puertos Ethernet WiFi dual para servicio FTTH residencial", "Huawei", 46.00),
    ("RA-002", "RA-DUP-002", "Terminal óptico Nokia GPON",
     "ONT Nokia 1GE básico para terminación de red GPON en domicilio abonado", "Nokia", 31.00),
    ("RA-007", "RA-DUP-003", "Switch L2 24 puertos GE Cisco",
     "Switch no gestionado 24 GE uplink SFP+ Cisco para armario distribución", "Cisco", 950.00),
    ("RD-001", "RD-DUP-001", "AAU Huawei 5G NR 3.5GHz 64T64R",
     "Unidad activa de antena 5G NR banda n78 3.5GHz 64T64R para sitio macro", "Huawei", 19200.00),
    ("RD-002", "RD-DUP-002", "Radio Ericsson AIR 5G 3.5GHz",
     "Unidad radio 5G New Radio Ericsson AIR6449 64 antenas integradas 3.5GHz", "Ericsson", 20000.00),
    ("RD-006", "RD-DUP-003", "Antena sectorial tri-banda pasiva 65 grados",
     "Antena multi-banda 700/1800/2600MHz 65 grados apertura 18dBi para macro", "Kathrein", 1950.00),
    ("SV-001", "SV-DUP-001", "Servidor rack HPE ProLiant DL380 Gen10",
     "Servidor 2U HPE DL380G10 con 2 procesadores Xeon y 32GB para virtualización", "HPE", 8800.00),
    ("SV-002", "SV-DUP-002", "Servidor Dell R740 2 sockets Xeon",
     "Servidor PowerEdge R740 Dell doble socket Xeon Gold memoria 64GB SSD 480", "Dell", 11800.00),
    ("SV-005", "SV-DUP-003", "SSD 480GB SATA servidor HPE 2.5",
     "Unidad de estado sólido HPE 480GB SATA 2.5 pulgadas hot swap ProLiant", "HPE", 195.00),
    ("CL-001", "CL-DUP-001", "Unidad refrigeración precisión Stulz 30kW",
     "CRAC 30kW Stulz para sala técnica o CPD con regulación inteligente", "Stulz", 29500.00),
    ("CL-002", "CL-DUP-002", "SAI trifásico APC 80kVA CPD",
     "Sistema UPS trifásico APC Galaxy VM 80kVA para alimentación ininterrumpida CPD", "APC", 44000.00),
    ("HE-001", "HE-DUP-001", "Fusionadora FO Fujikura 62S Plus",
     "Máquina de empalme fibra óptica Fujikura 62S+ alineación automática de núcleo", "Fujikura", 3950.00),
    ("HE-003", "HE-DUP-002", "Casco de obra con visera abatible",
     "EPI casco protección con pantalla facial para trabajadores en campo Telco", "MSA", 48.00),
    ("OC-001", "OC-DUP-001", "Tubo corrugado doble pared 110mm PVC",
     "Canalización PVC doble pared diámetro 110mm para enterrado telecomunicaciones", "Dura-Line", 4.10),
    ("CU-001", "CU-DUP-001", "Cable red Cat6A 4 pares apantallado",
     "Cable UTP apantallado Cat6A 23AWG 4 pares para instalaciones Gigabit LAN", "Draka", 0.92),
    ("FO-013", "FO-DUP-005", "Cierre empalme exterior 24 FO",
     "Caja de empalme fibra óptica exterior para 24 fibras con sellado estanco IP68", "Corning", 49.00),
]

duplicate_items = []
for (sku_base, sku_dup, name_dup, desc_dup, sup_dup, price_dup) in duplicate_pairs:
    base = next(item for item in unique_items if item['sku'] == sku_base)
    dup = base.copy()
    dup.update({'sku': sku_dup, 'name': name_dup, 'description': desc_dup,
                'supplier': sup_dup, 'unit_price_eur': price_dup, 'is_duplicate': True})
    duplicate_items.append(dup)

for item in unique_items:
    item['is_duplicate'] = False

catalog_items = unique_items + duplicate_items
df_catalog = pd.DataFrame(catalog_items)
df_catalog = df_catalog.sample(frac=1, random_state=42).reset_index(drop=True)  # mezclar

df_catalog.to_csv(os.path.join(BASE, 'catalog.csv'), index=False)
print(f'catalog.csv generado: {len(df_catalog)} artículos ({df_catalog.is_duplicate.sum()} duplicados)')
df_catalog.groupby('category').size()

catalog.csv generado: 200 artículos (20 duplicados)


category
cables_cobre          15
climatizacion_cpd     16
equipos_red_acceso    27
equipos_red_core      14
fibra_optica          47
herramientas_epi      18
obra_civil            15
radio_acceso          27
servidores_it         21
dtype: int64

## 2. Contratos marco — `contracts.json`

5 contratos marco vigentes con proveedores principales de la Telco.

In [5]:
contracts = [
    {
        "contract_id": "2024-CM-FO-001",
        "title": "Contrato Marco Fibra Óptica y Accesorios — Corning & Prysmian",
        "suppliers": [
            {"name": "Corning", "market_share_target": 0.60},
            {"name": "Prysmian", "market_share_target": 0.40}
        ],
        "categories": ["fibra_optica"],
        "negotiated_prices": {
            "FO-001": {"Corning": 1.65, "Prysmian": 1.72},
            "FO-002": {"Corning": 2.60, "Prysmian": 2.68},
            "FO-003": {"Prysmian": 3.90, "Corning": 3.98},
            "FO-007": {"Prysmian": 0.38, "Corning": 0.39},
        },
        "validity_start": "2024-01-01",
        "validity_end": "2025-12-31",
        "lead_time_days": 5,
        "min_order_quantity": 500,
        "volume_discounts": [
            {"min_km": 0, "max_km": 100, "discount_pct": 0},
            {"min_km": 100, "max_km": 500, "discount_pct": 3},
            {"min_km": 500, "max_km": 99999, "discount_pct": 6}
        ],
        "key_terms": "Pedido mínimo 500 metros por línea. Entrega en almacén Madrid o Barcelona. Penalización por retraso 0.5% por día hasta máximo 10%. Garantía de calidad certificada ITU-T G.652D.",
        "sla_delivery_compliance_pct": 0.96
    },
    {
        "contract_id": "2024-CM-RA-001",
        "title": "Contrato Marco Equipos Radio Acceso 4G/5G — Huawei & Ericsson",
        "suppliers": [
            {"name": "Huawei", "market_share_target": 0.55},
            {"name": "Ericsson", "market_share_target": 0.45}
        ],
        "categories": ["radio_acceso"],
        "negotiated_prices": {
            "RD-001": {"Huawei": 16500.00},
            "RD-002": {"Ericsson": 17200.00},
            "RD-003": {"Huawei": 10800.00},
            "RD-004": {"Ericsson": 12200.00},
        },
        "validity_start": "2024-01-01",
        "validity_end": "2026-12-31",
        "lead_time_days": 21,
        "min_order_quantity": 1,
        "volume_discounts": [
            {"min_units": 0, "max_units": 10, "discount_pct": 0},
            {"min_units": 10, "max_units": 50, "discount_pct": 2},
            {"min_units": 50, "max_units": 99999, "discount_pct": 5}
        ],
        "key_terms": "Incluye instalación y puesta en servicio. Soporte 24x7 incluido primer año. Firmware actualizado a última versión certificada. Plazo de entrega 21 días naturales desde OC.",
        "sla_delivery_compliance_pct": 0.92
    },
    {
        "contract_id": "2024-CM-IT-001",
        "title": "Contrato Marco Servidores y Hardware IT — HPE & Dell",
        "suppliers": [
            {"name": "HPE", "market_share_target": 0.65},
            {"name": "Dell", "market_share_target": 0.35}
        ],
        "categories": ["servidores_it"],
        "negotiated_prices": {
            "SV-001": {"HPE": 7800.00},
            "SV-002": {"Dell": 10200.00},
            "SV-003": {"HPE": 5200.00},
            "SV-004": {"HPE": 16500.00},
        },
        "validity_start": "2023-07-01",
        "validity_end": "2025-06-30",
        "lead_time_days": 14,
        "min_order_quantity": 1,
        "volume_discounts": [
            {"min_units": 0, "max_units": 5, "discount_pct": 0},
            {"min_units": 5, "max_units": 20, "discount_pct": 4},
            {"min_units": 20, "max_units": 99999, "discount_pct": 8}
        ],
        "key_terms": "Garantía hardware 3 años Next Business Day on-site. Instalación incluida para pedidos mayores de 10 unidades. Precio fijo durante vigencia del contrato.",
        "sla_delivery_compliance_pct": 0.98
    },
    {
        "contract_id": "2024-CM-FTTH-001",
        "title": "Contrato Marco ONTs y Equipos Acceso FTTH — Huawei & Nokia & ZTE",
        "suppliers": [
            {"name": "Huawei", "market_share_target": 0.50},
            {"name": "Nokia", "market_share_target": 0.30},
            {"name": "ZTE", "market_share_target": 0.20}
        ],
        "categories": ["equipos_red_acceso"],
        "negotiated_prices": {
            "RA-001": {"Huawei": 38.00},
            "RA-002": {"Nokia": 25.00},
            "RA-003": {"Huawei": 50.00},
            "RA-009": {"ZTE": 31.00},
        },
        "validity_start": "2024-04-01",
        "validity_end": "2026-03-31",
        "lead_time_days": 10,
        "min_order_quantity": 100,
        "volume_discounts": [
            {"min_units": 0, "max_units": 500, "discount_pct": 0},
            {"min_units": 500, "max_units": 2000, "discount_pct": 3},
            {"min_units": 2000, "max_units": 99999, "discount_pct": 7}
        ],
        "key_terms": "Pedido mínimo 100 unidades por línea de pedido. Entrega en almacén logístico Madrid. Las ONTs deben incluir firmware homologado por el operador. Embalaje individual en caja protectora.",
        "sla_delivery_compliance_pct": 0.94
    },
    {
        "contract_id": "2024-CM-OC-001",
        "title": "Contrato Marco Obra Civil y Canalización — Dura-Line & Prehorsa",
        "suppliers": [
            {"name": "Dura-Line", "market_share_target": 0.60},
            {"name": "Prehorsa", "market_share_target": 0.40}
        ],
        "categories": ["obra_civil"],
        "negotiated_prices": {
            "OC-001": {"Tubos Agua": 3.50, "Dura-Line": 3.45},
            "OC-002": {"Dura-Line": 1.75},
            "OC-003": {"Prehorsa": 255.00},
            "OC-004": {"Tubos Reunidos": 390.00},
        },
        "validity_start": "2024-01-01",
        "validity_end": "2025-12-31",
        "lead_time_days": 7,
        "min_order_quantity": 100,
        "volume_discounts": [
            {"min_units": 0, "max_units": 1000, "discount_pct": 0},
            {"min_units": 1000, "max_units": 5000, "discount_pct": 2.5},
            {"min_units": 5000, "max_units": 99999, "discount_pct": 5}
        ],
        "key_terms": "Pedido mínimo 100 unidades/metros. Entrega obra o almacén según acuerdo. Los tubos cumplen norma UNE-EN 50086. Tubos marcados con metrado para control en obra.",
        "sla_delivery_compliance_pct": 0.91
    }
]

with open(os.path.join(BASE, 'contracts.json'), 'w', encoding='utf-8') as f:
    json.dump(contracts, f, ensure_ascii=False, indent=2)

print(f'contracts.json generado: {len(contracts)} contratos marco')
for c in contracts:
    print(f"  {c['contract_id']}: {c['title'][:60]}...")

contracts.json generado: 5 contratos marco
  2024-CM-FO-001: Contrato Marco Fibra Óptica y Accesorios — Corning & Prysmia...
  2024-CM-RA-001: Contrato Marco Equipos Radio Acceso 4G/5G — Huawei & Ericsso...
  2024-CM-IT-001: Contrato Marco Servidores y Hardware IT — HPE & Dell...
  2024-CM-FTTH-001: Contrato Marco ONTs y Equipos Acceso FTTH — Huawei & Nokia &...
  2024-CM-OC-001: Contrato Marco Obra Civil y Canalización — Dura-Line & Preho...


## 3. Transacciones históricas — `transactions.csv`

2000 transacciones de compra del último año, con varios compradores y señales realistas:  
- maverick spend (compras fuera de contrato)
- desviaciones de cuota
- sobreprecios puntuales

In [6]:
buyers = [
    {"id": "buyer_mad_001", "name": "Ana García", "region": "Madrid", "zone": "norte"},
    {"id": "buyer_mad_002", "name": "Carlos Ruiz", "region": "Madrid", "zone": "sur"},
    {"id": "buyer_bcn_001", "name": "Marta Puig", "region": "Barcelona", "zone": "litoral"},
    {"id": "buyer_vlc_001", "name": "José Martínez", "region": "Valencia", "zone": "centro"},
    {"id": "buyer_svq_001", "name": "Laura Fernández", "region": "Sevilla", "zone": "andalucia"},
]

# Categorías con sus SKUs válidos (sin duplicados)
valid_skus_by_cat = df_catalog[~df_catalog.is_duplicate].groupby('category')['sku'].apply(list).to_dict()

# Parámetros de volumen por categoría
cat_params = {
    "fibra_optica":       {"qty_range": (500, 50000), "freq": 0.25},
    "equipos_red_acceso": {"qty_range": (10, 500),    "freq": 0.20},
    "radio_acceso":       {"qty_range": (1, 20),      "freq": 0.12},
    "equipos_red_core":   {"qty_range": (1, 10),      "freq": 0.08},
    "servidores_it":      {"qty_range": (1, 15),      "freq": 0.10},
    "climatizacion_cpd":  {"qty_range": (1, 5),       "freq": 0.06},
    "herramientas_epi":   {"qty_range": (1, 50),      "freq": 0.08},
    "obra_civil":         {"qty_range": (100, 5000),  "freq": 0.07},
    "cables_cobre":       {"qty_range": (50, 5000),   "freq": 0.04},
}
categories = list(cat_params.keys())
cat_freqs = [cat_params[c]['freq'] for c in categories]
cat_freqs_norm = [f/sum(cat_freqs) for f in cat_freqs]

# Generar transacciones
transactions = []
start_date = datetime(2024, 1, 1)

for i in range(2000):
    buyer = random.choice(buyers)
    category = np.random.choice(categories, p=cat_freqs_norm)
    skus_available = valid_skus_by_cat.get(category, [])
    if not skus_available:
        continue
    sku = random.choice(skus_available)
    item = df_catalog[df_catalog.sku == sku].iloc[0]

    qty_min, qty_max = cat_params[category]['qty_range']
    quantity = int(np.random.lognormal(mean=np.log((qty_min+qty_max)/2), sigma=0.5))
    quantity = max(qty_min, min(qty_max, quantity))

    # Precio: mayoritariamente cercano al catálogo, con desviaciones
    is_maverick = random.random() < 0.12  # 12% de compras fuera de contrato
    price_factor = np.random.normal(1.0, 0.05)
    if is_maverick:
        price_factor *= np.random.uniform(1.08, 1.25)  # sobreprecio maverick

    unit_price = round(item['unit_price_eur'] * price_factor, 4)
    unit_price = max(unit_price, 0.01)
    total_eur = round(unit_price * quantity, 2)

    # Fecha aleatoria en 2024
    days_offset = int(np.random.uniform(0, 365))
    order_date = start_date + timedelta(days=days_offset)

    # Proveedor: con probabilidad refleja cuotas del contrato
    cat_supplier_shares = {
        "fibra_optica":       {"Corning": 0.62, "Prysmian": 0.30, "Fujikura": 0.08},
        "radio_acceso":       {"Huawei": 0.58, "Ericsson": 0.35, "Nokia": 0.04, "Kathrein": 0.02, "Huber+Suhner": 0.01},
        "equipos_red_acceso": {"Huawei": 0.52, "Nokia": 0.28, "ZTE": 0.12, "Cisco": 0.06, "HPE": 0.02},
        "servidores_it":      {"HPE": 0.68, "Dell": 0.25, "VMware": 0.07},
        "climatizacion_cpd":  {"Stulz": 0.45, "APC": 0.40, "Eaton": 0.10, "Legrand": 0.05},
        "equipos_red_core":   {"Cisco": 0.45, "Juniper": 0.30, "Huawei": 0.15, "Nokia": 0.10},
        "cables_cobre":       {"Prysmian": 0.75, "Draka": 0.25},
        "obra_civil":         {"Dura-Line": 0.40, "Prehorsa": 0.30, "Tubos Agua": 0.20, "Tubos Reunidos": 0.10},
        "herramientas_epi":   {"Fujikura": 0.25, "Fluke": 0.25, "MSA": 0.30, "Yokogawa": 0.10, "Peli": 0.10},
    }
    supplier_dist = cat_supplier_shares.get(category, {item['supplier']: 1.0})
    suppliers_list = list(supplier_dist.keys())
    supplier_probs = list(supplier_dist.values())
    supplier = np.random.choice(suppliers_list, p=supplier_probs)

    transactions.append({
        "transaction_id": f"TXN-{i+1:05d}",
        "order_date": order_date.strftime("%Y-%m-%d"),
        "month": order_date.strftime("%Y-%m"),
        "buyer_id": buyer['id'],
        "buyer_name": buyer['name'],
        "region": buyer['region'],
        "sku": sku,
        "item_name": item['name'],
        "category": category,
        "supplier": supplier,
        "quantity": quantity,
        "uom": item['uom'],
        "unit_price_eur": unit_price,
        "total_eur": total_eur,
        "catalog_price_eur": item['unit_price_eur'],
        "price_deviation_pct": round((unit_price / item['unit_price_eur'] - 1) * 100, 2),
        "is_maverick": is_maverick,
        "has_contract": category in ["fibra_optica", "radio_acceso", "servidores_it", "equipos_red_acceso", "obra_civil"],
    })

df_transactions = pd.DataFrame(transactions)
df_transactions.to_csv(os.path.join(BASE, 'transactions.csv'), index=False)

print(f'transactions.csv generado: {len(df_transactions)} transacciones')
print(f'  Gasto total: {df_transactions.total_eur.sum():,.0f} EUR')
print(f'  Maverick spend: {df_transactions[df_transactions.is_maverick].total_eur.sum():,.0f} EUR ({df_transactions.is_maverick.mean()*100:.1f}%)')
print(f'  Período: {df_transactions.order_date.min()} a {df_transactions.order_date.max()}')

transactions.csv generado: 2000 transacciones
  Gasto total: 2,423,704,649 EUR
  Maverick spend: 150,559,135 EUR (11.2%)
  Período: 2024-01-01 a 2024-12-30


## 4. Forecast presupuestario — `forecast.csv`

Forecast mensual por comprador y categoría para 2024-2025.

In [7]:
# Calcular gasto real mensual por comprador + categoría como base del forecast
real_monthly = df_transactions.groupby(['buyer_id', 'month', 'category'])['total_eur'].sum().reset_index()

forecast_rows = []
months_2024 = pd.date_range('2024-01', '2024-12', freq='MS').strftime('%Y-%m').tolist()
months_2025 = pd.date_range('2025-01', '2025-12', freq='MS').strftime('%Y-%m').tolist()

# Presupuesto base por comprador y categoría (EUR/mes)
budget_base = {
    "fibra_optica":       {"buyer_mad_001": 280000, "buyer_mad_002": 200000, "buyer_bcn_001": 250000, "buyer_vlc_001": 150000, "buyer_svq_001": 120000},
    "radio_acceso":       {"buyer_mad_001": 350000, "buyer_mad_002": 280000, "buyer_bcn_001": 320000, "buyer_vlc_001": 200000, "buyer_svq_001": 180000},
    "equipos_red_acceso": {"buyer_mad_001": 120000, "buyer_mad_002": 90000,  "buyer_bcn_001": 110000, "buyer_vlc_001": 70000,  "buyer_svq_001": 60000},
    "servidores_it":      {"buyer_mad_001": 90000,  "buyer_mad_002": 60000,  "buyer_bcn_001": 80000,  "buyer_vlc_001": 50000,  "buyer_svq_001": 40000},
    "equipos_red_core":   {"buyer_mad_001": 200000, "buyer_mad_002": 150000, "buyer_bcn_001": 180000, "buyer_vlc_001": 100000, "buyer_svq_001": 80000},
    "climatizacion_cpd":  {"buyer_mad_001": 50000,  "buyer_mad_002": 35000,  "buyer_bcn_001": 45000,  "buyer_vlc_001": 25000,  "buyer_svq_001": 20000},
    "herramientas_epi":   {"buyer_mad_001": 15000,  "buyer_mad_002": 12000,  "buyer_bcn_001": 13000,  "buyer_vlc_001": 8000,   "buyer_svq_001": 7000},
    "obra_civil":         {"buyer_mad_001": 40000,  "buyer_mad_002": 30000,  "buyer_bcn_001": 35000,  "buyer_vlc_001": 20000,  "buyer_svq_001": 18000},
    "cables_cobre":       {"buyer_mad_001": 12000,  "buyer_mad_002": 9000,   "buyer_bcn_001": 10000,  "buyer_vlc_001": 7000,   "buyer_svq_001": 5000},
}

# Estacionalidad: más gasto en Q2 y Q3 (despliegues de verano)
seasonality = {
    '01': 0.75, '02': 0.78, '03': 0.88, '04': 1.05, '05': 1.12, '06': 1.20,
    '07': 1.18, '08': 0.80, '09': 1.15, '10': 1.10, '11': 0.95, '12': 1.04
}

for month in months_2024 + months_2025:
    month_num = month.split('-')[1]
    season_factor = seasonality[month_num]
    year = int(month.split('-')[0])
    year_growth = 1.08 if year == 2025 else 1.0  # crecimiento 8% en 2025

    for category, buyer_budgets in budget_base.items():
        for buyer_id, base_eur in buyer_budgets.items():
            forecast_eur = round(base_eur * season_factor * year_growth * np.random.uniform(0.95, 1.05), 2)
            forecast_rows.append({
                "month": month,
                "buyer_id": buyer_id,
                "category": category,
                "forecast_eur": forecast_eur,
                "budget_base_eur": base_eur,
                "season_factor": season_factor,
            })

df_forecast = pd.DataFrame(forecast_rows)
df_forecast.to_csv(os.path.join(BASE, 'forecast.csv'), index=False)

print(f'forecast.csv generado: {len(df_forecast)} filas')
print(f'  Presupuesto total 2024: {df_forecast[df_forecast.month.str.startswith("2024")].forecast_eur.sum():,.0f} EUR')
print(f'  Presupuesto total 2025: {df_forecast[df_forecast.month.str.startswith("2025")].forecast_eur.sum():,.0f} EUR')

forecast.csv generado: 1080 filas
  Presupuesto total 2024: 50,629,521 EUR
  Presupuesto total 2025: 54,809,962 EUR


## 5. Textos de contratos para RAG — `contracts/*.txt`

Documentos de texto que el agente indexará con ChromaDB para responder preguntas sobre condiciones contractuales.

In [8]:
contract_texts = {
    "2024-CM-FO-001.txt": """CONTRATO MARCO DE SUMINISTRO DE FIBRA ÓPTICA Y ACCESORIOS
Referencia: 2024-CM-FO-001
Fecha: 1 de enero de 2024
Vigencia: 1 enero 2024 — 31 diciembre 2025

PROVEEDORES HOMOLOGADOS Y CUOTAS DE MERCADO
- Corning Iberia S.L. — Cuota objetivo: 60%
- Prysmian Cables y Sistemas S.A. — Cuota objetivo: 40%

CATEGORÍAS CUBIERTAS
Fibra óptica monomodo y multimodo, cables de acometida FTTH, accesorios de empalme y conectores, splitters ópticos, repartidores ópticos (ODF), microductos HDPE.

PRECIOS NEGOCIADOS (EUR/metro, sin IVA)
- Cable FO SM G.652D 12H exterior: Corning 1,65 EUR/m | Prysmian 1,72 EUR/m
- Cable FO SM G.652D 24H exterior: Corning 2,60 EUR/m | Prysmian 2,68 EUR/m
- Cable FO SM G.652D 48H exterior: Prysmian 3,90 EUR/m | Corning 3,98 EUR/m
- Cable drop FTTH 2FO: Prysmian 0,38 EUR/m | Corning 0,39 EUR/m

DESCUENTOS POR VOLUMEN
- 0 a 100 km/pedido: sin descuento adicional
- 100 a 500 km/pedido: 3% descuento sobre tarifa
- Más de 500 km/pedido: 6% descuento sobre tarifa

CONDICIONES DE ENTREGA
- Pedido mínimo: 500 metros por línea de pedido
- Plazo de entrega estándar: 5 días hábiles desde confirmación de orden
- Entrega en almacén logístico de Madrid (Getafe) o Barcelona (Zona Franca)
- El transporte está incluido en el precio para pedidos superiores a 2.000 EUR

CALIDAD Y CERTIFICACIONES
- Todos los cables deben cumplir ITU-T G.652D y IEC 60794-1
- Certificado de calidad por bobina incluido en el albarán
- Garantía de fabricación: 25 años

PENALIZACIONES
- Retraso en entrega: 0,5% del valor del pedido por día natural de retraso
- Máximo penalización: 10% del valor total del pedido
- Defectos de calidad: reposición gratuita en plazo máximo 10 días

SEGUIMIENTO DE CUOTA
La cuota de cada proveedor se revisa trimestralmente. Desviaciones superiores al 10% del objetivo activarán el protocolo de reequilibrio en los siguientes 2 meses.

CONTACTO COMERCIAL
- Corning: Pedro Alonso (palonso@corning.com) — tel. 91 555 12 34
- Prysmian: Isabel Torres (i.torres@prysmian.com) — tel. 93 444 56 78
""",

    "2024-CM-RA-001.txt": """CONTRATO MARCO DE EQUIPOS DE RADIO ACCESO 4G/5G
Referencia: 2024-CM-RA-001
Fecha: 1 de enero de 2024
Vigencia: 1 enero 2024 — 31 diciembre 2026

PROVEEDORES Y CUOTAS
- Huawei Technologies España S.L. — Cuota objetivo: 55%
- Ericsson España S.A. — Cuota objetivo: 45%

ALCANCE
Unidades de radio remotas (RRU/AAU) para bandas 700MHz, 1800MHz, 2600MHz y 3.5GHz. Unidades de banda base (BBU). Small cells indoor y outdoor. Antenas sectoriales pasivas. Accesorios de radioenlace (cables feeder, conectores DIN 7/16).

PRECIOS NEGOCIADOS (EUR/unidad, sin IVA)
- AAU 5G NR 64T64R 3.5GHz Huawei: 16.500 EUR
- AIR 5G NR 64T64R 3.5GHz Ericsson: 17.200 EUR
- BBU 5G/4G Huawei BBU5900: 10.800 EUR
- Baseband 5G Ericsson BB6630: 12.200 EUR

DESCUENTOS POR VOLUMEN (unidades por pedido)
- 1 a 9 unidades: precio base
- 10 a 49 unidades: -2% sobre precio base
- 50 o más unidades: -5% sobre precio base

CONDICIONES
- Plazo de entrega: 21 días naturales desde emisión de orden de compra
- La instalación y puesta en servicio están incluidas en el precio
- El soporte técnico 24x7x365 está incluido durante el primer año
- El firmware debe estar actualizado a la última versión certificada por el operador
- Garantía hardware: 3 años a partir de la fecha de aceptación

INTEGRACIÓN
Los equipos deben integrarse con el sistema de gestión de red (EMS/NMS) del operador. El proveedor proporcionará soporte de integración sin cargo adicional durante los primeros 6 meses.

SEGURIDAD
Todos los equipos han superado la auditoría de ciberseguridad del operador. Acceso remoto únicamente mediante VPN corporativa con autenticación de doble factor.
""",

    "2024-CM-IT-001.txt": """CONTRATO MARCO DE SERVIDORES Y HARDWARE IT
Referencia: 2024-CM-IT-001
Vigencia: 1 julio 2023 — 30 junio 2025

PROVEEDORES Y CUOTAS
- Hewlett Packard Enterprise (HPE) — Cuota objetivo: 65%
- Dell Technologies Iberia S.L. — Cuota objetivo: 35%

ALCANCE
Servidores rack 1U y 2U, servidores blade, sistemas de almacenamiento SAN/NAS, switches de red para CPD, componentes (RAM, discos SSD/HDD, módulos ópticos), licencias de software de gestión.

PRECIOS NEGOCIADOS PRINCIPALES
- HPE ProLiant DL380 Gen10 (2xXeon): 7.800 EUR
- Dell PowerEdge R740 (2xXeon Gold): 10.200 EUR
- HPE ProLiant DL360 Gen10 (1xXeon): 5.200 EUR
- HPE MSA 2060 SAN: 16.500 EUR

CONDICIONES DE GARANTÍA Y SOPORTE
- Garantía hardware: 3 años Next Business Day on-site
- Tiempo de respuesta NBD para averías críticas (P1): 4 horas
- La instalación y racking están incluidos para pedidos ≥10 unidades
- Acceso al portal de soporte online 24x7

ENTREGA
- Plazo estándar: 14 días hábiles desde confirmación de pedido
- Configuración a medida (BTO): puede extender el plazo hasta 21 días
- Los precios son fijos durante toda la vigencia del contrato

RENOVACIÓN
Este contrato vence el 30 de junio de 2025. La negociación del nuevo contrato debe iniciarse antes del 1 de marzo de 2025.
""",

    "2024-CM-FTTH-001.txt": """CONTRATO MARCO DE ONTs Y EQUIPOS DE ACCESO FTTH
Referencia: 2024-CM-FTTH-001
Vigencia: 1 abril 2024 — 31 marzo 2026

PROVEEDORES Y CUOTAS
- Huawei Technologies España S.L. — Cuota objetivo: 50%
- Nokia Solutions and Networks Spain S.L. — Cuota objetivo: 30%
- ZTE Spain S.L. — Cuota objetivo: 20%

ALCANCE
Terminales de red óptica (ONT/ONU) para servicio residencial y PYME, líneas terminales ópticas (OLT), tarjetas de línea GPON y XGS-PON, switches de acceso y distribución para DSLAM.

PRECIOS NEGOCIADOS (EUR/unidad)
- ONT Huawei EG8145V5 4GE+WiFi: 38,00 EUR
- ONT Nokia G-010G-P 1GE: 25,00 EUR
- ONT Huawei EG8247H5 4GE+WiFi: 50,00 EUR
- ONT ZTE F670L 4GE+WiFi: 31,00 EUR

CONDICIONES LOGÍSTICAS
- Pedido mínimo: 100 unidades por línea de pedido
- Plazo de entrega: 10 días hábiles
- Entrega única en almacén logístico de Madrid (Getafe)
- Las ONTs deben incluir firmware en versión homologada por el operador
- Embalaje individual en caja con ESD protection

REQUISITOS TÉCNICOS
- Compatibilidad certificada con la plataforma GPON/XGS-PON del operador
- Las ONTs deben soportar TR-069 para gestión remota
- Etiquetado con número de serie y MAC en código de barras y QR
""",

    "2024-CM-OC-001.txt": """CONTRATO MARCO DE OBRA CIVIL Y CANALIZACIÓN
Referencia: 2024-CM-OC-001
Vigencia: 1 enero 2024 — 31 diciembre 2025

PROVEEDORES Y CUOTAS
- Dura-Line Iberia S.L. — Cuota objetivo: 60%
- Prehorsa Prefabricados Hormigón S.A. — Cuota objetivo: 40%

ALCANCE
Tubos de canalización PVC y PEAD para telecomunicaciones, microductos HDPE para soplado de fibra, arquetas prefabricadas de hormigón, tapas de registro, postes metálicos galvanizados, cinta señalizadora.

PRECIOS (EUR/metro o EUR/unidad según producto)
- Tubo PVC corrugado DN110: Tubos Agua 3,50 EUR | Dura-Line 3,45 EUR
- Tubo PEAD doble pared 63mm: Dura-Line 1,75 EUR
- Arqueta hormigón tipo H: Prehorsa 255,00 EUR
- Poste galvanizado 9m: Tubos Reunidos 390,00 EUR

CONDICIONES
- Pedido mínimo: 100 metros o unidades por línea
- Plazo de entrega: 7 días hábiles
- Entrega en obra o en almacén según acuerdo en cada pedido
- Los tubos cumplen norma UNE-EN 50086 y están marcados con metrado
- Las arquetas cumplen NTE-IER/1973 y tienen certificado de carga
"""
}

contracts_dir = os.path.join(BASE, 'contracts')
for filename, content in contract_texts.items():
    filepath = os.path.join(contracts_dir, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f'  Escrito: {filename} ({len(content)} chars)')

print(f'\nContratos TXT generados: {len(contract_texts)} ficheros')

  Escrito: 2024-CM-FO-001.txt (2018 chars)
  Escrito: 2024-CM-RA-001.txt (1624 chars)
  Escrito: 2024-CM-IT-001.txt (1248 chars)
  Escrito: 2024-CM-FTTH-001.txt (1175 chars)
  Escrito: 2024-CM-OC-001.txt (1011 chars)

Contratos TXT generados: 5 ficheros


## 6. Resumen y validación

In [9]:
print('=' * 60)
print('RESUMEN DE DATOS GENERADOS')
print('=' * 60)

print(f'\ncatalog.csv')
print(f'  Total artículos:  {len(df_catalog)}')
print(f'  Artículos únicos: {(~df_catalog.is_duplicate).sum()}')
print(f'  Duplicados:       {df_catalog.is_duplicate.sum()}')
print(f'  Categorías:       {df_catalog.category.nunique()}')
print(f'  Proveedores:      {df_catalog.supplier.nunique()}')

print(f'\ntransactions.csv')
print(f'  Total transacciones: {len(df_transactions)}')
print(f'  Gasto total:         {df_transactions.total_eur.sum():>14,.0f} EUR')
print(f'  Maverick spend:      {df_transactions[df_transactions.is_maverick].total_eur.sum():>14,.0f} EUR ({df_transactions.is_maverick.mean()*100:.1f}%)')
print(f'  Compradores:         {df_transactions.buyer_id.nunique()}')
print(f'  Período:             {df_transactions.order_date.min()} → {df_transactions.order_date.max()}')

print(f'\nforecast.csv')
print(f'  Filas:               {len(df_forecast)}')
print(f'  Presupuesto 2024:    {df_forecast[df_forecast.month.str.startswith("2024")].forecast_eur.sum():>14,.0f} EUR')
print(f'  Presupuesto 2025:    {df_forecast[df_forecast.month.str.startswith("2025")].forecast_eur.sum():>14,.0f} EUR')

print(f'\ncontracts.json:       {len(contracts)} contratos marco')
print(f'contracts/*.txt:      {len(contract_texts)} documentos para RAG')

print('\n✓ Todos los archivos guardados en data/synthetic/')

# Validación rápida
assert len(df_catalog) == 200, 'El catálogo debe tener exactamente 200 artículos'
assert df_catalog.is_duplicate.sum() == 20, 'Deben existir exactamente 20 duplicados'
assert len(df_transactions) == 2000, 'Deben existir exactamente 2000 transacciones'
print('\n✓ Todas las validaciones OK')

RESUMEN DE DATOS GENERADOS

catalog.csv
  Total artículos:  200
  Artículos únicos: 180
  Duplicados:       20
  Categorías:       9
  Proveedores:      27

transactions.csv
  Total transacciones: 2000
  Gasto total:          2,423,704,649 EUR
  Maverick spend:         150,559,135 EUR (11.2%)
  Compradores:         5
  Período:             2024-01-01 → 2024-12-30

forecast.csv
  Filas:               1080
  Presupuesto 2024:        50,629,521 EUR
  Presupuesto 2025:        54,809,962 EUR

contracts.json:       5 contratos marco
contracts/*.txt:      5 documentos para RAG

✓ Todos los archivos guardados en data/synthetic/

✓ Todas las validaciones OK


In [10]:
# Vista previa del catálogo
print('Distribución por categoría:')
print(df_catalog[~df_catalog.is_duplicate].groupby('category').agg(
    n_items=('sku', 'count'),
    precio_min=('unit_price_eur', 'min'),
    precio_max=('unit_price_eur', 'max'),
    precio_medio=('unit_price_eur', 'mean')
).round(2).to_string())

print('\nTop 5 categorías por gasto real (transactions):')
print(df_transactions.groupby('category')['total_eur'].sum().sort_values(ascending=False).head(5).apply(lambda x: f'{x:,.0f} EUR'))

Distribución por categoría:
                    n_items  precio_min  precio_max  precio_medio
category                                                         
cables_cobre             14        0.48        4.50          2.08
climatizacion_cpd        14     1200.00    52000.00      19395.84
equipos_red_acceso       24       27.38     5200.00       1883.38
equipos_red_core         14       48.00    68000.00      24618.34
fibra_optica             42        0.26     3809.99        206.67
herramientas_epi         16       38.00    13673.96       3417.98
obra_civil               14        0.14      420.00        111.36
radio_acceso             24       18.50    19200.00       7336.45
servidores_it            18      167.77    29519.11       8295.06

Top 5 categorías por gasto real (transactions):
category
fibra_optica          2,093,301,857 EUR
equipos_red_acceso      197,886,725 EUR
obra_civil               55,527,783 EUR
equipos_red_core         20,898,650 EUR
radio_acceso             18,